In [12]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

print("🚀 Booting Elite Data Fusion Engine (Level: Utter Perfection)...")

# 1. Load Primary Trip Data
features_df = pd.read_csv("../data/raw/delhi_traffic_features.csv")
target_df = pd.read_csv("../data/raw/delhi_traffic_target.csv")
master_df = pd.merge(features_df, target_df, on="Trip_ID", how="inner")

# 2. Load Auxiliary Data Lakes
rides_df = pd.read_csv("../data/raw/ncr_rides_final_event_dataset_2024.csv")
vanet_df = pd.read_csv("../data/raw/vanet_traffic_data.csv")
stops_df = pd.read_csv("../data/raw/stops.csv")
aqi_df = pd.read_csv("../data/raw/RS_Session_262_AU_2113_1.csv")

print("✅ All Auxiliary Data Lakes Loaded Successfully.")

🚀 Booting Elite Data Fusion Engine (Level: Utter Perfection)...
✅ All Auxiliary Data Lakes Loaded Successfully.


In [13]:
# 1. Cyclical Time Math
time_map = {'Morning Peak': 8, 'Afternoon': 14, 'Evening Peak': 18, 'Night': 2}
day_map = {'Weekday': 2, 'Weekend': 6}

master_df['hour_estimate'] = master_df['time_of_day'].map(time_map).fillna(12)
master_df['hour_sin'] = np.sin(2 * np.pi * master_df['hour_estimate'] / 24.0)
master_df['hour_cos'] = np.cos(2 * np.pi * master_df['hour_estimate'] / 24.0)

master_df['day_estimate'] = master_df['day_of_week'].map(day_map).fillna(3)
master_df['day_sin'] = np.sin(2 * np.pi * master_df['day_estimate'] / 7.0)
master_df['day_cos'] = np.cos(2 * np.pi * master_df['day_estimate'] / 7.0)

# 2. Uber Surge/Demand Proxy
rides_df['surge_proxy'] = rides_df['Booking Value'] / (rides_df['Ride Distance'] + 0.1)
area_friction = rides_df.groupby('Pickup Location').agg(
    historical_surge_multiplier=('surge_proxy', 'mean'),
    historical_wait_time=('Avg VTAT', 'mean')
).reset_index()

master_df = pd.merge(master_df, area_friction, left_on='start_area', right_on='Pickup Location', how='left')
master_df['historical_surge_multiplier'] = master_df['historical_surge_multiplier'].fillna(master_df['historical_surge_multiplier'].median())
master_df['historical_wait_time'] = master_df['historical_wait_time'].fillna(master_df['historical_wait_time'].median())
master_df.drop(columns=['Pickup Location'], inplace=True, errors='ignore')

print("✅ Base Intelligence & Cyclical Math Injected.")

✅ Base Intelligence & Cyclical Math Injected.


In [14]:
print("📡 Extracting Micro-Telemetry from VANET...")

# Convert VANET timestamps to hours
vanet_df['timestamp'] = pd.to_datetime(vanet_df['timestamp'], errors='coerce')
vanet_df['hour'] = vanet_df['timestamp'].dt.hour

# Map VANET hours to your specific time_of_day categories
def map_hour_to_tod(hour):
    if 7 <= hour < 11: return 'Morning Peak'
    elif 11 <= hour < 16: return 'Afternoon'
    elif 16 <= hour < 20: return 'Evening Peak'
    else: return 'Night'

vanet_df['time_of_day'] = vanet_df['hour'].apply(map_hour_to_tod)

# Calculate average queue lengths and communication delays per time bucket
vanet_friction = vanet_df.groupby('time_of_day').agg(
    vanet_avg_queue_length=('queue_length_veh', 'mean'),
    vanet_comm_delay_ms=('avg_comm_delay_ms', 'mean')
).reset_index()

# Merge into master dataset
master_df = pd.merge(master_df, vanet_friction, on='time_of_day', how='left')

# Fill any weird gaps with median
master_df['vanet_avg_queue_length'] = master_df['vanet_avg_queue_length'].fillna(master_df['vanet_avg_queue_length'].median())
master_df['vanet_comm_delay_ms'] = master_df['vanet_comm_delay_ms'].fillna(master_df['vanet_comm_delay_ms'].median())

print("✅ VANET Micro-Telemetry Fused!")

📡 Extracting Micro-Telemetry from VANET...
✅ VANET Micro-Telemetry Fused!


In [15]:
print("🚇 Calculating Local Infrastructure Density...")

stops_df['stop_name'] = stops_df['stop_name'].astype(str).str.lower()

def count_stops_in_area(area_name):
    if pd.isna(area_name): return 0
    area_name_lower = str(area_name).lower()
    # Count how many times this area appears in the stop names
    return stops_df['stop_name'].str.contains(area_name_lower).sum()

# Calculate density for both start and end areas
master_df['start_stop_density'] = master_df['start_area'].apply(count_stops_in_area)
master_df['end_stop_density'] = master_df['end_area'].apply(count_stops_in_area)

# Total route infrastructure complexity
master_df['route_total_stops'] = master_df['start_stop_density'] + master_df['end_stop_density']

print("✅ Infrastructure (Metro/Bus Stop) Density Mapped!")

🚇 Calculating Local Infrastructure Density...
✅ Infrastructure (Metro/Bus Stop) Density Mapped!


In [16]:
print("🌫️ Mapping Air Quality Index (AQI) Penalties...")

# Extract Delhi's "Severe" days ratio from the AQI dataset
delhi_aqi = aqi_df[aqi_df['City'].str.contains('Delhi', case=False, na=False)]

# Since we don't have specific dates, we use weather conditions as an environmental proxy
# In Delhi, Fog/Smog heavily correlates with Severe AQI and drastic speed reductions
def get_aqi_penalty(weather):
    weather = str(weather).lower()
    if 'fog' in weather or 'smog' in weather:
        return 1.45  # 45% friction penalty for Severe AQI visibility
    elif 'rain' in weather:
        return 1.30  # 30% friction penalty for slippery roads
    else:
        return 1.0   # Baseline

master_df['environmental_friction_penalty'] = master_df['weather_condition'].apply(get_aqi_penalty)

# Finally, One-Hot Encode the remaining text categories so the AI can read them
categorical_cols = ['weather_condition', 'traffic_density_level', 'road_type']
for col in categorical_cols:
    if col in master_df.columns:
        dummies = pd.get_dummies(master_df[col], prefix=col, drop_first=True).astype(int)
        master_df = pd.concat([master_df, dummies], axis=1)

# Drop original text columns
columns_to_drop = ['start_area', 'end_area', 'time_of_day', 'day_of_week', 'weather_condition', 'traffic_density_level', 'road_type', 'hour_estimate', 'day_estimate']
master_df.drop(columns=[col for col in columns_to_drop if col in master_df.columns], inplace=True)

print("✅ AQI Environmental Data Fused & Text Cleaned.")

🌫️ Mapping Air Quality Index (AQI) Penalties...
✅ AQI Environmental Data Fused & Text Cleaned.


In [17]:
# Move the target column to the very end
target_col = 'travel_time_minutes'
cols = [c for c in master_df.columns if c != target_col] + [target_col]
master_df = master_df[cols]

# Save
os.makedirs("../data/processed", exist_ok=True)
save_path = "../data/processed/master_training_data.csv"
master_df.to_csv(save_path, index=False)

print(f"🏆 ABSOLUTE PERFECTION ACHIEVED!")
print(f"Fused Features: {master_df.shape[1]}")
print(f"Saved to: {save_path}")

🏆 ABSOLUTE PERFECTION ACHIEVED!
Fused Features: 24
Saved to: ../data/processed/master_training_data.csv
